In [ ]:
# imports

import os
import re
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')


if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    



In [ ]:
# Connect to client libraries

openai = OpenAI()

ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
clients = {"gpt-5": openai, "gpt-oss:20b": ollama}
models = list(clients)

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [ ]:
system_prompt = """
You add short, helpful comments to source code.
Do not change, remove or reorder any existing code. Only add comments, either on their own line or at the end of a line.
Keep comments brief and focused on what the code does. Avoid large comment blocks.
Respond only with the commented code: no explanations and no markdown fences.
"""

def user_prompt_for(code, language):
    return f"""Add comments to this {language} code using {language} comment syntax.

{code}
"""

In [ ]:
def messages_for(code, language):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(code, language)}
    ]

In [ ]:
def strip_fences(reply):
    # Models often wrap code in ```lang ... ``` despite being told not to; also works on partial streamed replies
    return re.sub(r"^```[\w+-]*\n|\n?```\s*$", "", reply.strip())

def code_unchanged(original, commented):
    # Every original line must start some output line, in order (end-of-line comments allowed)
    out = (line.strip() for line in commented.splitlines())
    return all(any(o.startswith(line) for o in out) for line in (l.strip() for l in original.splitlines()) if line)

def comment_code(model, code, language):
    reasoning_effort = "low" if model == "gpt-5" else None
    reply = ""
    try:
        stream = clients[model].chat.completions.create(model=model, messages=messages_for(code, language), reasoning_effort=reasoning_effort, stream=True)
        for chunk in stream:
            if chunk.choices:
                reply += chunk.choices[0].delta.content or ""
                yield strip_fences(reply), "Streaming..."
    except Exception as e:
        raise gr.Error(f"{model} failed: {e}")
    commented = strip_fences(reply)
    status = "✅ Original code preserved" if code_unchanged(code, commented) else "⚠️ The model changed the original code, review before using"
    yield commented, status

In [ ]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
languages = ["python", "c", "cpp", "javascript", "typescript", "html", "css", "sql", "shell", "r", "json", "yaml", "dockerfile", "markdown"]

def set_language(language):
    # Apply the chosen language to both panes
    return gr.Code(language=language, label=f"{language} (original)"), gr.Code(language=language, label=f"{language} (commented)")

with gr.Blocks(theme=gr.themes.Monochrome(), title="Comment your code out") as ui:
    with gr.Row():
        language = gr.Dropdown(languages, value=languages[0], label="Language")
        model = gr.Dropdown(models, value=models[0], label="Model")
        convert = gr.Button("Comment code")

    with gr.Row(equal_height=True):
        original = gr.Code(label=f"{languages[0]} (original)", value=pi, language=languages[0], lines=26)
        commented = gr.Code(label=f"{languages[0]} (commented)", value="", language=languages[0], lines=26)

    status = gr.Textbox(label="Check", interactive=False)

    language.change(fn=set_language, inputs=[language], outputs=[original, commented])
    convert.click(fn=comment_code, inputs=[model, original, language], outputs=[commented, status])

ui.launch(inbrowser=True)

In [ ]:
#gr.close_all()